In [48]:
from typing import TypedDict, NotRequired, Annotated, Sequence, List
from pydantic import BaseModel
from pandas import DataFrame
import yaml
from dotenv import load_dotenv

from langgraph.graph import START, END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, ToolMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_openrouter import ChatOpenRouter


In [50]:
load_dotenv()

True

In [57]:
class MultiAgentState(TypedDict):
    messages: NotRequired[Annotated[Sequence[BaseMessage], add_messages]]
    query: NotRequired[str]
    table_ids: NotRequired[List[str]]
    table_descriptions: NotRequired[List[str]]
    filepath: NotRequired[str]

    df: NotRequired[DataFrame]

class RAGOutput(BaseModel):
    table_ids: List[str]
    

In [64]:
main_agent_system_prompt = """
<summary>
Ты - агент для аналитиков, который упрощает отвечает на вопросы по данным из банковской Базы Данных. 
Твоя задача с помощью доступных тебе инструментов получить ответ на вопрос пользователя в виде пути к Excel-файлу с нужными данными.
</summary>

<tools>
Доступные тебе инструменты:
1. RAG tool - инструмент, который по запросу пользователя выдает id самых релевантных таблиц для ответа на вопрос.
2. Extract Table Descriptions - инструмент, который по переданному ему списку id таблиц вовзращает описание нужных таблиц.
3. SQL Agent - агент, который по запросу пользователя и описанию таблиц выдаст тебе путь до Excel-файла с нужными данными.
</tools>

<algoritm>
Алгоритм твоих действий:
1. Передать запрос пользователя в RAG tool и получить из него id таблиц Базы Данных.
2. Передать полученные на прошлом шаге id в инструмент Extract Table Descriptions и получить описание нужных таблиц.
3. Передать полученное описание SQL агенту, а после получить от него путь до файла с данными.
4. Отправить пользователю путь до полученного файла с данными.
</algoritm>
"""

In [40]:
@tool
def rag_tool(query: str) -> RAGOutput:
    """Найти нужные таблицы для ответа на запрос пользователя"""
    table_ids = 'customers'

    return {'table_ids': [table_ids]}

@tool
def extract_table_descriptions(table_ids: List[str]) -> dict:
    """Получение описания столбцов необходимых таблиц"""

    with open('db_schema.yaml') as table_description_file:
        schema = yaml.safe_load(table_description_file)

    table_descriptions = []
    
    for table in schema['tables']:
        if table['table_id'] in table_ids:
            table_descriptions.append(table)

    return {'table_descriptions': table_descriptions}

tools = [rag_tool, extract_table_descriptions]

In [ ]:
main_llm = ChatOpenRouter(
    model='qwen/qwen3-235b-a22b-2507'
).bind_tools(tools)

In [68]:
def model_call(state: MultiAgentState) -> dict:
    returned_dict = {}
    if len(state['messages']) == 0:
        query = input('Вы: ')
        returned_dict['query'] = query

        new_messages = [SystemMessage(content=main_agent_system_prompt)] + [HumanMessage(content=query)]
    else:
        new_messages = state['messages']
        
    response = main_llm.invoke(new_messages)
    returned_dict['messages'] = new_messages + [response]

    return returned_dict

def need_tools(state: MultiAgentState) -> str:
    last_message = state['messages'][-1]

    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return 'tool'
    return 'continue'

In [69]:
graph = StateGraph(MultiAgentState)

graph.add_node('model_call', model_call)
main_agent_tools_node = ToolNode(tools=tools)
graph.add_node('main_agent_tools', main_agent_tools_node)

graph.add_edge(START, 'model_call')
graph.add_conditional_edges(
    "model_call", 
    need_tools, 
    {
        'tool': 'main_agent_tools',
        'continue': END
    }
)
graph.add_edge('main_agent_tools', 'model_call')

app = graph.compile()

In [70]:
app.invoke({})

{'messages': [SystemMessage(content='\n<summary>\nТы - агент для аналитиков, который упрощает отвечает на вопросы по данным из банковской Базы Данных. \nТвоя задача с помощью доступных тебе инструментов получить ответ на вопрос пользователя в виде пути к Excel-файлу с нужными данными.\n</summary>\n\n<tools>\nДоступные тебе инструменты:\n1. RAG tool - инструмент, который по запросу пользователя выдает id самых релевантных таблиц для ответа на вопрос.\n2. Extract Table Descriptions - инструмент, который по переданному ему списку id таблиц вовзращает описание нужных таблиц.\n3. SQL Agent - агент, который по запросу пользователя и описанию таблиц выдаст тебе путь до Excel-файла с нужными данными.\n</tools>\n\n<algoritm>\nАлгоритм твоих действий:\n1. Передать запрос пользователя в RAG tool и получить из него id таблиц Базы Данных.\n2. Передать полученные на прошлом шаге id в инструмент Extract Table Descriptions и получить описание нужных таблиц.\n3. Передать полученное описание SQL агенту,